# 2. Cleaning
Tiếp nối từ `01_eda_before_cleaning.ipynb` — đọc lại checkpoint `raw_parsed.parquet`
thay vì ingest lại từ GCS.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

In [ ]:
df = pd.read_parquet("data/interim/raw_parsed.parquet")
print(f"Loaded checkpoint: {df.shape}")

## 4. Làm sạch & xử lý missing
Gộp toàn bộ các bước xử lý giá trị thiếu/outlier vào một mạch, theo đúng thứ tự phụ thuộc giữa các cột (vd: lọc `indfall` trước khi xử lý `age`, xử lý `nomprov` trước khi impute `renta` theo tỉnh).

### 4.1 Lọc khách hàng đã mất (`indfall`)
Khách hàng đã mất (`indfall == "S"`) sẽ không mua sản phẩm nào nữa trong tương
lai — giữ lại sẽ làm nhiễu dữ liệu cho bài toán dự đoán/khuyến nghị sản phẩm.
Loại các dòng này trước khi xử lý các cột khác (bao gồm cả age).

In [ ]:
print(df["indfall"].value_counts(dropna=False))

n_deceased_old = df.loc[(df["age"] > 100) & (df["indfall"] == "S")].shape[0]
print(f"Số dòng tuổi > 100 và đã mất: {n_deceased_old}")

n_before = len(df)
df = df[df["indfall"] != "S"].copy()
n_after = len(df)

print(f"Đã loại {n_before - n_after} dòng (indfall == 'S') — {((n_before - n_after) / n_before * 100):.2f}% dataset")
print(f"Shape sau khi lọc: {df.shape}")

### 4.2 Xử lý `age`
Có giá trị thiếu và outlier (tuổi quá nhỏ/quá lớn): thay
outlier bằng mean của nhóm tuổi hợp lý gần nhất, nhưng in rõ số lượng outlier để
kiểm chứng thay vì làm ngầm.

In [ ]:
n_young = int((df["age"] < 18).sum())
n_old = int((df["age"] > 100).sum())
print(f"Outliers: {n_young} tuổi < 18, {n_old} tuổi > 100")

mean_young = df.loc[(df.age >= 18) & (df.age <= 30), "age"].mean()
mean_mid = df.loc[(df.age >= 30) & (df.age <= 100), "age"].mean()

df.loc[df.age < 18, "age"] = mean_young
df.loc[df.age > 100, "age"] = mean_mid
df["age"] = df["age"].fillna(df["age"].mean())
df["age"] = df["age"].astype(int)

In [ ]:
sns.histplot(df["age"], bins=80, color="tomato")
plt.title("Age Distribution (after cleaning)")
plt.xlim(15, 100)
plt.show()

### 4.3 Xử lý `ind_nuevo` (khách hàng mới)

**Fix**: cột này được load với `dtype=str` (theo cấu hình đọc CSV ở trên), nhưng
pandas bản mới có kiểu `string` nghiêm ngặt hơn — gán thẳng số nguyên `1` vào cột
`str` sẽ ném `TypeError`. Convert sang numeric trước khi impute.

In [ ]:
df["ind_nuevo"] = pd.to_numeric(df["ind_nuevo"], errors="coerce")
n_missing = df["ind_nuevo"].isnull().sum()
print(f"Missing: {n_missing}")

if n_missing > 0:
    months_active = df.loc[df["ind_nuevo"].isnull(), :].groupby("ncodpers").size()
    print(f"Max số tháng active trong nhóm thiếu dữ liệu: {months_active.max()}")
    # Số tháng active thấp -> đúng là khách hàng mới
    df.loc[df["ind_nuevo"].isnull(), "ind_nuevo"] = 1

### 4.4 Xử lý `antiguedad` (thâm niên)
Cùng nhóm khách hàng thiếu `ind_nuevo` ở trên.

In [ ]:
df["antiguedad"] = pd.to_numeric(df["antiguedad"], errors="coerce")

# Tính thâm niên thật từ chênh lệch ngày (theo tháng, không tính theo ngày/30.44
# vì antiguedad gốc của Santander là số tháng lịch, tính kiểu này sẽ khớp)
antiguedad_calc = (
    (df["fecha_dato"].dt.year - df["fecha_alta"].dt.year) * 12
    + (df["fecha_dato"].dt.month - df["fecha_alta"].dt.month)
)

# So sánh với cột gốc để biết mức độ lệch (chỉ so trên phần antiguedad không null)
diff = (df["antiguedad"] - antiguedad_calc).dropna()
print(f"Số dòng lệch >1 tháng giữa antiguedad gốc và antiguedad tính từ ngày: {(diff.abs() > 1).sum()}")
print(diff.describe())

# Dùng bản tính từ ngày để fill missing (chính xác hơn fillna bằng min)
df.loc[df["antiguedad"].isnull(), "antiguedad"] = antiguedad_calc[df["antiguedad"].isnull()]

# Giá trị âm trong data gốc thực chất là lỗi biết trước của dataset này
# (placeholder kiểu -999999 cho khách mới) — nên thay bằng giá trị tính từ ngày
# thay vì clip cứng về 0, vì antiguedad_calc phản ánh đúng thâm niên thật
df.loc[df["antiguedad"] < 0, "antiguedad"] = antiguedad_calc[df["antiguedad"] < 0]

# Fallback cuối cùng nếu vẫn còn null (trường hợp fecha_alta cũng null)
df["antiguedad"] = df["antiguedad"].fillna(df["antiguedad"].median())

### 4.5 Xử lý `fecha_alta`
Một số dòng thiếu ngày gia nhập — gán bằng giá trị trung vị (median date).

In [ ]:
if df["fecha_alta"].isnull().any():
    mask = df["fecha_alta"].isnull() & df["antiguedad"].notnull()

    # Trừ theo tháng lịch (to_period("M") - int) rồi convert lại về timestamp,
    # vectorized nên nhanh hơn nhiều so với apply() theo từng dòng
    fecha_alta_calc = (
        df.loc[mask, "fecha_dato"].dt.to_period("M")
        - df.loc[mask, "antiguedad"].round().astype(int)
    ).dt.to_timestamp()

    df.loc[mask, "fecha_alta"] = fecha_alta_calc

    # Fallback cuối: dòng nào antiguedad cũng thiếu luôn thì mới dùng median date như cũ
    still_missing = df["fecha_alta"].isnull()
    if still_missing.any():
        median_date = df["fecha_alta"].dropna().median()
        df.loc[still_missing, "fecha_alta"] = median_date
        print(f"Vẫn dùng median date cho {still_missing.sum()} dòng thiếu cả antiguedad")

### 4.6 Xử lý `indrel`
Fill giá trị thiếu bằng trạng thái phổ biến nhất (mode).

In [ ]:
print(df["indrel"].value_counts(dropna=False))

most_common = df["indrel"].mode(dropna=True)
fill_value = most_common.iloc[0] if len(most_common) else 1
df.loc[df["indrel"].isnull(), "indrel"] = fill_value

### 4.7 Loại bỏ cột không cần thiết
`tipodom` không hữu ích, `cod_prov` dư thừa vì đã có tên tỉnh ở `nomprov`, `conyuemp` vì quá nhiều null.

In [ ]:
df = df.drop(columns=["tipodom", "cod_prov", "conyuemp"], errors="ignore")
df.isnull().sum()[df.isnull().sum() > 0]

### 4.8 Xử lý `ind_actividad_cliente`
Fill bằng 1.

In [ ]:
df.loc[df["ind_actividad_cliente"].isnull(), "ind_actividad_cliente"] = 1

### 4.9 Xử lý `nomprov` (tên tỉnh)

**Fix**: bản gốc dùng chuỗi byte kiểu Python 2 (`"CORU\xc3\x91A, A"`) để sửa lỗi
encode của "CORUÑA" — cách này không match được trong Python 3. Sửa bằng ký tự
unicode `\u00d1` trực tiếp.

In [ ]:
df["nomprov"] = df["nomprov"].replace({"CORU\u00d1A, A": "CORUNA, A"})
df.loc[df["nomprov"].isnull(), "nomprov"] = "UNKNOWN"
df["nomprov"].unique()

### 4.10 Xử lý `renta` (thu nhập)

Thu nhập biến động nhiều theo tỉnh, nên impute theo median của từng tỉnh sẽ chính
xác hơn median toàn cục. Trực quan hoá trước:

In [ ]:
income_by_province = (
    df.loc[df["renta"].notnull()].groupby("nomprov")["renta"].median().sort_values()
)

plt.figure(figsize=(12, 6))
sns.barplot(x=income_by_province.index, y=income_by_province.values, color="#c60b1e")
plt.xticks(rotation=90)
plt.ylabel("Median Income")
plt.xlabel("Province")
plt.title("Income Distribution by Province")
plt.tight_layout()
plt.show()

**Fix bug**: bản gốc impute bằng cách `merge()` rồi gán qua `.reset_index()`,
làm lệch index giữa hai DataFrame nên gán nhầm giá trị cho nhiều hàng. Sửa bằng
`groupby().transform("median")` — đảm bảo alignment luôn đúng theo index gốc.

In [ ]:
province_median = df.groupby("nomprov")["renta"].transform("median")
df["renta"] = df["renta"].fillna(province_median)

# fallback nếu cả tỉnh không có giá trị nào để tính median
df["renta"] = df["renta"].fillna(df["renta"].median())

### 4.11 Xử lý `ind_nomina_ult1` / `ind_nom_pens_ult1`

Vì đây là time-series theo từng khách hàng theo
tháng, forward-fill theo lịch sử `ncodpers`

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"])
for col in ["ind_nomina_ult1", "ind_nom_pens_ult1"]:
    df[col] = df.groupby("ncodpers")[col].transform(lambda s: s.ffill())
    df[col] = df[col].fillna(0)

### 4.12 Các cột dạng chuỗi còn thiếu
Điền "UNKNOWN" hoặc giá trị hợp lý nhất tuỳ theo ý nghĩa từng cột.

In [ ]:
string_cols = df.select_dtypes(include=["object", "string"]).columns
missing_cols = [c for c in string_cols if df[c].isnull().any()]
print("Các cột còn thiếu:", missing_cols)

if "indfall" in missing_cols:
    df.loc[df["indfall"].isnull(), "indfall"] = "N"
if "tiprel_1mes" in missing_cols:
    df.loc[df["tiprel_1mes"].isnull(), "tiprel_1mes"] = "A"
    df["tiprel_1mes"] = df["tiprel_1mes"].astype("category")

if "indrel_1mes" in df.columns:
    map_dict = {
        1.0: "1", "1.0": "1", "1": "1",
        3.0: "3", "3.0": "3", "3": "3",
        2.0: "2", "2.0": "2", "2": "2",
        4.0: "4", "4.0": "4", "4": "4",
        "P": "P",
    }
    df["indrel_1mes"] = df["indrel_1mes"].fillna("P")
    df["indrel_1mes"] = df["indrel_1mes"].map(lambda x: map_dict.get(x, x))
    df["indrel_1mes"] = df["indrel_1mes"].astype("category")

remaining = [c for c in missing_cols if c not in ("indfall", "tiprel_1mes", "indrel_1mes")]
for col in remaining:
    df.loc[df[col].isnull(), col] = "UNKNOWN"

df.isnull().sum()[df.isnull().sum() > 0]

### 4.13 Kiểm tra lại — xác nhận hết missing

In [ ]:
remaining_na = df.isnull().sum()[df.isnull().sum() > 0]
remaining_na if len(remaining_na) else "Không còn cột nào thiếu dữ liệu."

### 4.14 Convert các cột sản phẩm (`ind_*_ult1`) sang kiểu int

In [ ]:
feature_cols = df.filter(regex="ind_.*ult.*").columns
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

feature_cols

## Checkpoint — lưu cho các notebook tiếp theo
Lưu `df` đã làm sạch để `03_eda_after_cleaning.ipynb` và `04_feature_engineering.ipynb`
đọc tiếp — cả 2 notebook đó đều nhánh ra từ đúng bản dữ liệu sạch này.

In [ ]:
df.to_parquet("data/interim/cleaned.parquet", index=False)
print(f"Đã lưu checkpoint: {df.shape}")